# Real-Time Multimodal Emotion Understanding - MELD Lean Build

**Track:** all three modalities (text + audio + vision). **All inference is local.**

| Component | Model | ~Params | Source |
|---|---|---|---|
| Text encoder | DistilRoBERTa (frozen) | 82M | pretrained HF |
| Audio encoder | Wav2Vec2-base (frozen) | 94M | pretrained HF |
| Face landmarker | MediaPipe FaceLandmarker (frozen) | ~2M (upper-bound est.) | pretrained |
| Fusion + 7-way emotion head | 2-layer transformer, trained here | ~2M | trained here |
| Response generator | Qwen2.5-1.5B-Instruct | 1.54B | pretrained HF |
| Prosody + face geometry features | librosa / arithmetic | 0 | handcrafted |
| **Total** | | **~1.72B / 6B budget** | |

**Definition of real-time for this prototype (turn-based interaction):**
- While a user turn is in progress, the engine receives streamed text chunks, audio windows and camera frames (`observe_*`).
- At turn end (`end_turn`), the structured emotion tags are returned with a target of **p95 <= 500 ms**; the generated reply streams afterwards with a target of **p95 <= 2000 ms** for the full text.
- Ablation: replacing any single modality with the learned absence token must degrade results gracefully, never crash.

In [ ]:
import sys, json, time
from pathlib import Path

# Repo root = folder containing pyproject.toml (works from repo root or notebooks/)
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import torch

from meld_robot import config, utils
from meld_robot.data import download_and_extract_meld, load_split, utterances_from_split, ensure_audio_wavs
from meld_robot.text_features import TextEncoder
from meld_robot.audio_features import AudioEncoder, prosody_features, prosody_evidence
from meld_robot.vision_features import FaceGeometryExtractor, sample_frames, vision_evidence
from meld_robot.model import FusionModel, MODALITY_NAMES
from meld_robot.train import featurize_split, MELDFusionDataset, train_model, evaluate
from meld_robot.engine import UtteranceEngine
from meld_robot.responder import PromptedResponder
from torch.utils.data import DataLoader

In [ ]:
utils.set_seed()
DEVICE = utils.get_device()
print("device:", DEVICE, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"gpu: {props.name} | {props.total_memory/1e9:.1f} GB")

## 1. Data

MELD comes from the official Hugging Face mirror `declare-lab/MELD` (`MELD.Raw.tar.gz`, ~10.9 GB: CSVs with dialogue utterances + per-utterance mp4 clips). Requires `ffmpeg` on PATH for audio extraction. **One-time setup: download (minutes to an hour depending on bandwidth) + wav extraction (~1 min) + featurization (see next section).**

In [ ]:
download_and_extract_meld()

train_df = load_split("train")
dev_df = load_split("dev")
train_utts = utterances_from_split("train")
dev_utts = utterances_from_split("dev")
test_utts = []
try:
    test_utts = utterances_from_split("test")  # labels may be missing in some copies
except Exception as exc:
    print("test split unavailable:", exc)

ensure_audio_wavs(train_utts)
ensure_audio_wavs(dev_utts)
if test_utts:
    ensure_audio_wavs(test_utts)

print(f"train: {len(train_utts)} | dev: {len(dev_utts)} | test: {len(test_utts)}")

counts = train_df["emotion"].value_counts().reindex(config.EMOTIONS)
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(counts.index, counts.values, color="#4C72B0")
ax.set_title("MELD train emotion distribution (heavily neutral-skewed -> class-weighted loss)")
ax.set_ylabel("utterances")
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Optional fast-pass: subsample training data for a quick first run.
# Set True to featurize only a subset (results will be worse; use for smoke-testing the loop).
USE_SUBSET = False
SUBSET_SIZE = 4000

if USE_SUBSET:
    rng = np.random.default_rng(config.SEED)
    keep = rng.choice(len(train_utts), size=min(SUBSET_SIZE, len(train_utts)), replace=False)
    train_utts = [train_utts[i] for i in keep]
    print(f"subset: {len(train_utts)} training utterances")

## 2. Featurization (one-time, cached)

Each utterance is reduced to:
- **text**: mean-pooled DistilRoBERTa last hidden states (768-d)
- **audio**: mean-over-time Wav2Vec2 last hidden states (768-d) + 8-d prosody vector (librosa: duration, loudness, pitch stats, spectral centroid, ZCR)
- **vision**: up to 8 evenly-sampled frames -> MediaPipe landmarks -> 12 interpretable geometry dims (eye aspect, brow raise, mouth open/width, corner pull, nostril widen, head pose) -> {mean, std, max} pooling + presence ratio (37-d)

Everything is cached to `data/feature_cache/<split>.npz`, so training iterations afterward are pure fusion-model work. **Expect roughly 0.2-1 s per utterance for vision; budget ~1-2 h for the full dataset, or set `USE_SUBSET=True` for a first pass.**

In [ ]:
text_encoder = TextEncoder(device=DEVICE)
audio_encoder = AudioEncoder(device=DEVICE)
face_extractor = FaceGeometryExtractor()

for split, utts in (("train", train_utts), ("dev", dev_utts), ("test", test_utts)):
    if not utts:
        continue
    featurize_split(split, text_encoder, audio_encoder, face_extractor, utts)

## 3. Fusion model training

Only the ~2M-parameter fusion transformer + emotion head are trained (everything else frozen). Key robustness decisions:
- **Modality dropout (p=0.2):** each sample randomly loses whole modalities during training, teaching the model to rely on what is actually present.
- **Class weighting:** inverse-frequency weights counter the ~48% neutral skew.
- **Selection metric:** dev weighted-F1 (the MELD benchmark metric), early stopping with patience 6.

In [ ]:
labels_of = lambda utts: {u.uid: config.EMOTION_TO_IDX[u.emotion] for u in utts}

train_ds = MELDFusionDataset(config.FEATURE_CACHE / "train.npz", labels_of(train_utts))
dev_ds = MELDFusionDataset(config.FEATURE_CACHE / "dev.npz", labels_of(dev_utts))

fusion_model, history = train_model(train_ds, dev_ds, device=DEVICE)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history["train_loss"], label="train")
ax1.plot(history["val_loss"], label="val")
ax1.set_title("cross-entropy loss"); ax1.set_xlabel("epoch"); ax1.legend()
ax2.plot(history["val_wf1"], color="green")
ax2.set_title("dev weighted-F1"); ax2.set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
dev_metrics = evaluate(fusion_model, DataLoader(dev_ds, batch_size=128), DEVICE)
print(f"dev weighted-F1 {dev_metrics['wf1']:.4f} | macro-F1 {dev_metrics['macro_f1']:.4f} | acc {dev_metrics['accuracy']:.4f}")

cm = np.array(dev_metrics["confusion"])
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im)
ax.set_xticks(range(7)); ax.set_xticklabels(config.EMOTIONS, rotation=30)
ax.set_yticks(range(7)); ax.set_yticklabels(config.EMOTIONS)
ax.set_xlabel("predicted"); ax.set_ylabel("gold")
ax.set_title("dev confusion matrix")
plt.show()

display(pd.DataFrame(dev_metrics["report"]).T.round(3))

if test_utts:
    test_ds = MELDFusionDataset(config.FEATURE_CACHE / "test.npz", labels_of(test_utts))
    test_metrics = evaluate(fusion_model, DataLoader(test_ds, batch_size=128), DEVICE)
    print(f"test weighted-F1 {test_metrics['wf1']:.4f} | macro-F1 {test_metrics['macro_f1']:.4f}")

## 4. Real-time demo: one utterance through the whole system

We replay a held-out MELD clip **as if it were live**: text streams in word by word, audio in three windows, camera frames as they arrive. Then `end_turn()` fuses everything into structured state, and the responder streams a grounded reply. The two outputs required by the assignment:
1. **Structured tags:** MELD emotion category + confidence + evidence + per-stage latency.
2. **Short response text grounded in the multimodal input** (it must cite at least two modalities).

In [ ]:
responder = PromptedResponder(device=DEVICE)
engine = UtteranceEngine(text_encoder, audio_encoder, face_extractor, fusion_model,
                         responder=responder, device=DEVICE)

pick = next((u for u in dev_utts if u.emotion in ("anger", "sadness", "disgust") and u.video_path), None) \
       or next((u for u in dev_utts if u.emotion != "neutral" and u.video_path), dev_utts[0])
print(f"utterance {pick.uid} | gold: {pick.emotion} | speaker: {pick.speaker}")
print(f"text: {pick.text!r}")

wav, sr = sf.read(pick.wav_path, dtype="float32")
frames = sample_frames(pick.video_path)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(np.linspace(0, len(wav)/sr, len(wav)), wav, lw=0.5)
ax1.set_title("audio streamed to the engine"); ax1.set_xlabel("s")
import cv2
if frames:
    ax2.imshow(cv2.cvtColor(frames[len(frames)//2], cv2.COLOR_BGR2RGB)); ax2.axis("off")
    ax2.set_title("a frame streamed to the engine")
plt.show()

In [ ]:
# --- stream the turn into the engine as it would happen live ---
engine.start_turn()

for i, w in enumerate(pick.text.split(" ")):
    engine.observe_text(w + (" " if i < len(pick.text.split(" ")) - 1 else ""))
for chunk in np.array_split(wav, 3):
    engine.observe_audio(chunk.astype(np.float32), sr=sr)
for f in frames:
    engine.observe_frame(f)

state = engine.end_turn()
print(json.dumps(state, indent=2))

In [ ]:
# --- streamed, grounded reply ---
t0 = time.perf_counter()
tokens = []
print("robot:", end=" ", flush=True)
for tok in engine.respond(state):
    print(tok, end="", flush=True)
    tokens.append(tok)
dt = (time.perf_counter() - t0) * 1000
print(f"\n[streamed {len(tokens)} tokens in {dt:.0f} ms | tag latency was {state['latency_ms']['total_ms']:.0f} ms]")

### Ablation: missing modality must not break the loop

Turn 2 uses the **same audio but no transcript and no camera** (text + vision absent). The fusion model's learned absence tokens handle it, the engine reports what was missing, and the responder still produces a coherent (more conservative) reply.

In [ ]:
engine.start_turn()
engine.observe_audio(wav.astype(np.float32), sr=sr)  # speech only

state2 = engine.end_turn()
print("emotion:", state2["emotion"], "| confidence:", state2["confidence"])
print("modalities present:", state2["modalities_present"])
print("top3:", state2["top3"])

print("robot:", end=" ", flush=True)
for tok in engine.respond(state2):
    print(tok, end="", flush=True)
print()

## 5. Parameter budget + hardware report

Every learned parameter in the local inference path is counted (total, not trainable/active). The MediaPipe landmarker ships as a task file without a published parameter count; we conservatively estimate ~2M from its 3.7 MB fp16 size and count it in full.

In [ ]:
budget_rows = [
    utils.BudgetRow("Text encoder - DistilRoBERTa (frozen)", text_encoder.param_count, "pretrained hf"),
    utils.BudgetRow("Audio encoder - Wav2Vec2-base (frozen)", audio_encoder.param_count, "pretrained hf"),
    utils.BudgetRow("Face landmarker - MediaPipe (frozen)", config.FACE_LANDMARKER_PARAMS, "pretrained mediapipe (upper-bound est.)"),
    utils.BudgetRow("Fusion transformer + emotion head (trained here)", utils.count_parameters(fusion_model), "trained here"),
    utils.BudgetRow("Response generator - Qwen2.5-1.5B-Instruct", responder.param_count, "pretrained hf"),
    utils.BudgetRow("Prosody + face geometry features", 0, "handcrafted (0 params)"),
]
budget = utils.parameter_table(budget_rows)  # raises if the 6B budget is exceeded
budget_df = pd.DataFrame(budget)
display(budget_df)

total = budget[-1]["params"]
print(f"TOTAL: {total:,} / {config.PARAM_BUDGET:,} = {100*total/config.PARAM_BUDGET:.1f}% of the 6B budget")

utils.save_json(
    {
        "budget": budget,
        "hardware": utils.hardware_report(),
        "history": history,
        "dev_metrics": {k: v for k, v in dev_metrics.items() if k not in ("preds", "gold")},
        "demo_state": state,
        "ablation_state": {k: state2[k] for k in ("emotion", "confidence", "top3", "modalities_present")},
        "local_inference_only": False,  # record whether any remote call was used
    },
    config.ARTIFACTS_DIR / "report.json",
)
print("saved:", config.ARTIFACTS_DIR / "report.json")

## Design decisions, trade-offs, limitations

**Decisions.** (1) Frozen specialist encoders + a tiny trained fusion: with ~10K training utterances, fine-tuning 82M+ encoders risks overfitting and triples the accounting surface; the 2M fusion is where cross-modal interaction is learned. (2) Prosody and face geometry feed *both* the model and human-readable evidence, keeping outputs grounded and auditable, and cost 0 parameters. (3) The responder only ever sees structured state, never raw signal - tiny prompt, tiny model, provably grounded output.

**Real-time.** Turn-based; the windowing happens during the turn (frames/audio buffered, features computed at turn end), tag target p95 500 ms, full reply target p95 2 s, measured per stage (see `latency_ms` in the demo state). On a single consumer GPU the Wav2Vec2+DistilRoBERTa+fusion pass is ~100-300 ms; Qwen 1.5B autoregressive decoding dominates the reply latency, which is why the reply streams.

**Known limitations.** (1) MELD audio contains canned laughter and music: models can partially learn `laughter -> joy` shortcuts that do not transfer to a robot microphone. (2) Vision features are expressive but shallow - no temporal dynamics beyond std/max pooling. (3) If live speech is used, ASR enters the parameter budget; here transcripts come from the dataset, and the engine's `observe_text` accepts any transcript source. (4) Class imbalance and label noise cap weighted-F1 in the mid-50s with this feature class.

**Deliberately left out (scope).** Live mic/camera capture, VAD-based turn endpointing, ASR, RL fine-tuning, and fine-grained video temporal modeling. See README for the extension map.